# Human Development Index (HDI) Prediction
### ML-0027 | Training Notebook

In [ ]:
# Step 1: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('Libraries imported successfully!')

In [ ]:
# Step 2: Load Dataset
df = pd.read_csv('../Dataset/HDI.csv')
print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Step 3: Exploratory Data Analysis
print('Dataset Info:')
print(df.info())
print('\nDescriptive Statistics:')
df.describe()

In [ ]:
# Step 4: Check Missing Values
print('Missing Values:')
print(df.isnull().sum())

In [ ]:
# Step 5: Data Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Human Development Index - Data Visualization', fontsize=16, fontweight='bold')

# HDI Distribution
axes[0, 0].hist(df['HDI'], bins=20, color='steelblue', edgecolor='white')
axes[0, 0].set_title('HDI Distribution')
axes[0, 0].set_xlabel('HDI')
axes[0, 0].set_ylabel('Frequency')

# Life Expectancy vs HDI
axes[0, 1].scatter(df['Life_Expectancy'], df['HDI'], alpha=0.6, color='darkorange')
axes[0, 1].set_title('Life Expectancy vs HDI')
axes[0, 1].set_xlabel('Life Expectancy')
axes[0, 1].set_ylabel('HDI')

# Education Index vs HDI
axes[1, 0].scatter(df['Education_Index'], df['HDI'], alpha=0.6, color='green')
axes[1, 0].set_title('Education Index vs HDI')
axes[1, 0].set_xlabel('Education Index')
axes[1, 0].set_ylabel('HDI')

# GNI per Capita vs HDI
axes[1, 1].scatter(df['GNI_per_Capita'], df['HDI'], alpha=0.6, color='purple')
axes[1, 1].set_title('GNI per Capita vs HDI')
axes[1, 1].set_xlabel('GNI per Capita (USD)')
axes[1, 1].set_ylabel('HDI')

plt.tight_layout()
plt.savefig('../Dataset/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plots saved!')

In [ ]:
# Step 6: Prepare Features and Target
X = df[['Life_Expectancy', 'Education_Index', 'GNI_per_Capita']]
y = df['HDI']

print('Features shape:', X.shape)
print('Target shape:', y.shape)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

In [ ]:
# Step 7: Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Scaling complete!')

In [ ]:
# Step 8: Train Multiple Models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results[name] = {'MAE': mae, 'MSE': mse, 'RMSE': np.sqrt(mse), 'R2': r2}
    print(f'{name}: MAE={mae:.4f}, RMSE={np.sqrt(mse):.4f}, R²={r2:.4f}')

In [ ]:
# Step 9: Model Comparison
results_df = pd.DataFrame(results).T
print('Model Comparison:')
print(results_df.round(4))

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results_df))
width = 0.25
ax.bar(x - width, results_df['MAE'], width, label='MAE', color='steelblue')
ax.bar(x, results_df['RMSE'], width, label='RMSE', color='darkorange')
ax.bar(x + width, results_df['R2'], width, label='R²', color='green')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_df.index)
ax.legend()
plt.tight_layout()
plt.savefig('../Dataset/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Step 10: Select Best Model & Save
# Random Forest typically performs best
best_model = models['Random Forest']
best_model.fit(X_train_scaled, y_train)

# Save model + scaler together as pipeline dict
model_bundle = {'model': best_model, 'scaler': scaler}
with open('../Flask/HDI.pkl', 'wb') as f:
    pickle.dump(model_bundle, f)

print('Model saved as HDI.pkl!')

# Verify saved model
with open('../Flask/HDI.pkl', 'rb') as f:
    loaded = pickle.load(f)

test_input = np.array([[72, 0.75, 15000]])
test_scaled = loaded['scaler'].transform(test_input)
pred = loaded['model'].predict(test_scaled)
print(f'Test prediction (Life=72, Edu=0.75, GNI=15000): HDI = {pred[0]:.3f}')

In [ ]:
# Step 11: Feature Importance
importances = best_model.feature_importances_
features = ['Life_Expectancy', 'Education_Index', 'GNI_per_Capita']

plt.figure(figsize=(8, 5))
plt.barh(features, importances, color=['steelblue', 'darkorange', 'green'])
plt.title('Feature Importance (Random Forest)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('../Dataset/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

for f, imp in zip(features, importances):
    print(f'{f}: {imp:.4f}')

## ✅ Training Complete!
Model saved at `../Flask/HDI.pkl`
